### Consolidating Data

In [22]:
import os
import sys

notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)

if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import date
from src.ingestion import bcb, tesouro
from src.curves import bootstrap, models, utils

# Avoids Restarting Kernel
%load_ext autoreload
%autoreload 2

# Data Load

data_bcb = bcb.fetch_all(start_date=date(2024, 1, 1),
                        include_focus=False) # Focus not responding
data_tesouro = tesouro.fetch_all(start_date=date(2024, 1, 1))

# Check BCB server for for VNA data

df_vna = data_bcb.get('vna_ntn_b')

if df_vna is None or df_vna.empty or "value" not in df_vna.columns:
    print("BCB API Down. Creating mock VNA baseline data for curve simulation.")
    
    # Extract all unique transaction dates from the Tesouro dataset
    unique_dates = pd.to_datetime(data_tesouro['ntnb_zero']['date']).unique()
    unique_dates = sorted(unique_dates)
    
    # Generate an approximate baseline VNA (e.g., around ~4,300 BRL for modern dates)
    mock_values = np.linspace(4100.0, 4400.0, len(unique_dates))

    df_vna = pd.DataFrame({
        'date': unique_dates,
        'value': mock_values
    })

    # Ensure date typing matches if the API actually is working
    df_vna['date'] = pd.to_datetime(df_vna['date'])

# Ensure Tesouro date column typing matches the mock VNA tracking table
df_tesouro_clean = data_tesouro['ntnb_zero'].copy()
df_tesouro_clean['date'] = pd.to_datetime(df_tesouro_clean['date'])

# 3. Execute the Bootstrap
df_ntnb = bootstrap.bootstrap_ntnb_principal(
    df_tesouro_clean, 
    df_vna
)

# Select date
target_date = df_ntnb['date'].max()
df_snapshot = df_ntnb[df_ntnb['date'] == target_date].sort_values('du')

print(f"Bootstrapped curve for: {target_date.date()}")
print(df_snapshot[['date', 'maturity', 'du', 'pu_base', 'yield_calculated']].head())

2026-06-09 15:52:14,320 [INFO] Fetching series 12. From 2024-01-01 to 2026-06-09 (attempt 1)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


2026-06-09 15:52:14,549 [INFO] Series 12: 611 records fetched.
2026-06-09 15:52:14,554 [INFO] Fetching series 433. From 2024-01-01 to 2026-06-09 (attempt 1)
2026-06-09 15:52:14,721 [INFO] Series 433: 28 records fetched.
2026-06-09 15:52:14,725 [INFO] Fetching series 226. From 2024-01-01 to 2026-06-09 (attempt 1)
2026-06-09 15:52:14,949 [INFO] Series 226: 907 records fetched.
2026-06-09 15:52:14,953 [INFO] Fetching series 432. From 2024-01-01 to 2026-06-09 (attempt 1)
2026-06-09 15:52:15,149 [INFO] Series 432: 891 records fetched.
2026-06-09 15:52:15,152 [INFO] Fetching series 11. From 2024-01-01 to 2026-06-09 (attempt 1)
2026-06-09 15:52:15,621 [INFO] Series 11: 611 records fetched.
2026-06-09 15:52:15,625 [INFO] Fetching series 12466. From 2024-01-01 to 2026-06-09 (attempt 1)
2026-06-09 15:52:15,968 [WARNING] HTTP error on attempt 1: 404 Client Error: Not Found for url: https://api.bcb.gov.br/dados/serie/bcdata.sgs.12466/dados?formato=json&dataInicial=01%2F01%2F2024&dataFinal=09%2F06%

BCB API Down. Creating mock VNA baseline data for curve simulation.
Bootstrapped curve for: 2026-06-08
           date   maturity    du  pu_base  yield_calculated
763  2026-06-08 2026-08-15    49  4627.38         -0.228277
1370 2026-06-08 2029-05-15   734  3714.93          0.059827
1455 2026-06-08 2032-08-15  1551  2873.82          0.071660
2062 2026-06-08 2035-05-15  2238  2364.78          0.072418
2396 2026-06-08 2040-08-15  3557  1644.57          0.072210


### Fitting Nelson-Siegel Model
Convert business days to year fraction and fit Model

In [25]:
# Time (t) in years and Yields (y)
t_obs = df_snapshot['du'].values / 252
y_obs = df_snapshot['yield_calculated'].values

# Fit parameters
b0, b1, b2, tau = models.fit_nelson_siegel(t_obs, y_obs)

print(f"NS Parameters:\n Level (b0): {b0:.3f}\n Slope (b1): {b1:.3f}\n Curvature (b2): {b2:.3f}\n Tau: {tau:.3f}")

NS Parameters:
 Level (b0): 0.071
 Slope (b1): -0.375
 Curvature (b2): 0.382
 Tau: 0.873


### Visualizing the Curve
How the parametric model smooths the market curve. 

In [26]:
# Generate continuous curve for plotting (0 to 10 years)
t_curve = np.linspace(0.1, 10, 100)
y_curve = models.nelson_siegel(t_curve, b0, b1, b2, tau)

fig = go.Figure()

# Market Points
fig.add_trace(go.Scatter(
    x=t_obs, y=y_obs, 
    mode='markers', 
    name='Market Data (NTN-B)',
    marker=dict(size=10, color='red')
))

# Nelson-Siegel Fit
fig.add_trace(go.Scatter(
    x=t_curve, y=y_curve, 
    mode='lines', 
    name='Nelson-Siegel Fit',
    line=dict(dash='dash', color='blue')
))

fig.update_layout(
    title=f"IPCA Real Yield Curve - {target_date.date()}",
    xaxis_title="Maturity (Years)",
    yaxis_title="Annual Real Yield (%)",
    template="plotly_white"
)

fig.show()